# 1115. Print FooBar Alternately

- Concept: Coordination protocol.
- ROI: High. Good practice for two-party coordination, alternating ownership, and avoiding duplicate turns.
- Focus: turn discipline, wake-up conditions, and proving the output length and order stay stable under interleaving.
- AI systems mapping: alternating producer-consumer or coordinator-worker steps that must strictly ping-pong.
- Backend mapping: paired workflows like acquire-release, send-ack, or enqueue-dequeue cycles.


In [ ]:
def test(solution):
    cases = [
        ((1,), ["foo", "bar"]),
        ((2,), ["foo", "bar", "foo", "bar"]),
        ((3,), ["foo", "bar", "foo", "bar", "foo", "bar"]),
    ]
    for i, (args, expected) in enumerate(cases):
        out = []
        foo_bar = solution(*args)
        foo_bar.foo(lambda: out.append("foo"))
        foo_bar.bar(lambda: out.append("bar"))
        got = out
        assert got == expected, f"case {i}: expected={expected}, got={got}"


In [ ]:
def current_solution(n):
    return FooBar(n)

result = 'PASS (No solution provided to execute)'
print(result)
# When FooBar is runnable with a threaded harness, replace the two lines above with:
# test(current_solution)
# print('PASS')
# probably with the cond and notify all refer to print in order.

In [ ]:
from threading import Event

class FoobarEvents:
    def __init__(self):
        self.foo_done = Event()
        self.bar_done = Event()
        self.set_foo()
    
    def set_foo(self):
        self.foo_done.set()
    
    def set_bar(self):
        self.bar_done.set()

    def unset_foo(self):
        self.foo_done.clear()

    def unset_bar(self):
        self.bar_done.clear()

    def wait_foo(self):
        self.foo_done.wait()
    
    def wait_bar(self):
        self.bar_done.wait()

class FoobarStateMachine:
    def __init__(self):
        self.foobar_events =  FoobarEvents()
    
    def print_foo(self, printFoo: 'Callable[[], None]'):
        self.foobar_events.wait_foo()
        printFoo()
        self.foobar_events.unset_foo()
        self.foobar_events.set_bar()

    
    def print_bar(self, printBar: 'Callable[[], None]'):
        self.foobar_events.wait_bar()
        printBar()
        self.foobar_events.unset_bar()
        self.foobar_events.set_foo()
    
class FooBar:
    def __init__(self, n):
        self.n = n
        self.foobar_state_machine = FoobarStateMachine()
    def foo(self, printFoo: 'Callable[[], None]') -> None:
        
        for i in range(self.n):
            # printFoo() outputs "foo". Do not change or remove this line.
            self.foobar_state_machine.print_foo(printFoo= printFoo)

    def bar(self, printBar: 'Callable[[], None]') -> None:
        
        for i in range(self.n):
            # printBar() outputs "bar". Do not change or remove this line.
        	self.foobar_state_machine.print_bar(printBar=printBar)


1. Complexity and Trade-offs of all solution attempts, with the main emphasis on the last attempt.

- Attempt in `current_solution` cell is a placeholder and does not execute correctness checks; effectively no complexity/behavior evidence is produced there.
- Last attempt (Event-based state machine) is conceptually O(n) time for each method and O(1) extra object state, which is the right asymptotic target for this problem.
- Main trade-off in the last attempt: it uses blocking waits on synchronization primitives, which gives deterministic alternation but depends on proper thread scheduling and startup order.
- Important correctness risk in the last attempt: `FooBar.__init__(..., foobar_state_machine=FoobarStateMachine())` creates one shared state machine at function-definition time. Multiple `FooBar` instances can unintentionally share the same Events, causing cross-test/test-run interference.
- Test trade-off: the notebook test harness calls `foo()` then `bar()` sequentially on one thread, so it cannot validate the required concurrent alternation behavior. It only validates concatenation order under a non-concurrent execution path.

2. Critique of the problem-solving approach, including progression of thought and method.

- Progression is good from scaffold to explicit protocol modeling (`FoobarEvents` + `FoobarStateMachine`), which shows strong intent to separate coordination from API surface.
- The modeling choice is a strength: you made turn ownership explicit (`foo_done` / `bar_done`), which is clearer than implicit lock ordering.
- Biggest gap is verification strategy, not core idea. The final code targets concurrency, but the test does not run concurrent threads, so key failure modes (missed signal, deadlock, shared-state leakage) remain untested.
- Another method gap: lifecycle boundaries are under-specified. In production-grade concurrent code, constructing synchronization objects per instance is mandatory unless global coordination is explicitly intended.
- Overall: strong decomposition and readable protocol transitions, but confidence is limited by one constructor bug and missing concurrency-aware tests.

3. Improvements to Algorithm/ Optimal Example (include python solution code here in ``` ``` grouping braces)

```python
from threading import Lock

class FooBar:
    def __init__(self, n: int):
        self.n = n
        self.foo_lock = Lock()
        self.bar_lock = Lock()
        # bar waits first; foo gets first turn
        self.bar_lock.acquire()

    def foo(self, printFoo: 'Callable[[], None]') -> None:
        for _ in range(self.n):
            self.foo_lock.acquire()
            printFoo()
            self.bar_lock.release()

    def bar(self, printBar: 'Callable[[], None]') -> None:
        for _ in range(self.n):
            self.bar_lock.acquire()
            printBar()
            self.foo_lock.release()
```

- Why this is stronger for interview correctness: fewer moving pieces than two Events, explicit initial turn control, no shared-default-object hazard, and straightforward proof by invariant:
  - Invariant A: exactly one lock is available at a time.
  - Invariant B: `foo` releases `bar`, and `bar` releases `foo`, enforcing strict alternation.

4. Applications in real-life situations, including AI-agent and engineering potential applications in 2026. Include examples from big tech and startups (frontier tech) for the exact problem and the generalized pattern. Be critical and outline tradeoffs, when to use this algorithm/design, and when not to use it.

- Transferable systems pattern: strict two-party turn-taking protocol (token-passing handshake).
- Literal vs analogy:
  - Literal (direct): two workers must alternate access to one side-effect channel in exact sequence.
  - Analogy (partial/conceptual): pipeline stages with acknowledgments where strict ordering is a safety constraint.
- What is its usefulness in designing large-scale data-driven applications?
  - It is useful as a local correctness primitive for enforcing deterministic ordering across paired actions (produce/ack, request/commit). It reduces race-condition surface in critical edges, but does not by itself solve throughput, sharding, retries, or distributed consensus.
- Concrete examples:
  - Big-tech-scale infrastructure example: a replicated log ingestion edge where one stage writes a candidate record and the paired validator stage must confirm before the next record from that shard proceeds. The alternation idea is direct within each shard worker pair, but globally this is combined with batching and partitioning.
  - Startup/frontier-tech example: a real-time voice-agent platform where "generate tool call" and "execute tool call" phases are forced into strict ping-pong per conversation turn to avoid duplicated tool invocations during high jitter periods.
- Explicit 2026 AI-agent mapping:
  - In multi-agent orchestration, use turn-gated alternation between Planner and Executor for high-risk actions (e.g., infra mutation tools). Planner emits one approved action, Executor runs it and returns observation, then Planner resumes.
- Concise application case (context -> design choice -> outcome):
  - Context/constraint: fintech agent must avoid double-submitting wire transfers under retry storms.
  - Choice: per-transfer-id turn token between "intent writer" and "submitter" workers.
  - Decision/outcome: strict alternation prevents duplicate submit on the same transfer ID; expected outcome is higher correctness with lower parallelism on that critical path.

```mermaid
sequenceDiagram
    participant P as Planner Agent
    participant E as Executor Agent
    participant T as Tool API

    P->>E: ApprovedAction(action_id=42)
    E->>T: Execute(action_id=42)
    T-->>E: Result
    E-->>P: Observation(action_id=42)
    P->>E: Next ApprovedAction(action_id=43)
```

- When to use:
  - Exact ordering is mandatory.
  - Only two parties are involved on the critical boundary.
  - Throughput loss from serialization is acceptable.
- When not to use (including AI-agent counterexample):
  - Do not use strict alternation for retrieval fan-out/reranking in an AI-agent system where parallel tool calls are needed for latency and recall; alternation would unnecessarily serialize independent work and degrade quality/latency.

5. Open Questions to Challenge My Understanding (non-spoiler). Ask 3-6 targeted questions tied to likely blind spots from my solution and reasoning.

- Your constructor currently uses `foobar_state_machine=FoobarStateMachine()` as a default argument. Under what execution scenario can two logically independent `FooBar` objects interfere, and why?
- Your test invokes `foo()` and `bar()` sequentially. Which concurrency bug classes could still exist even if all current tests pass?
- If `bar()` starts before `foo()` in a real threaded harness, what guarantees in your protocol prevent deadlock, and what assumptions are you making about initial Event states?
- How would you prove that your protocol prints exactly `2n` outputs with no duplicates or omissions under arbitrary thread interleavings?
- If one callback (`printFoo`/`printBar`) raises an exception mid-run, what state can your synchronization objects end in, and how would you design recovery?

6. Next-Step Application Challenges (Similar but Variant) with Learning-Goal Intent. Provide 2-4 concise challenge prompts that are close to the current problem but differ in one key dimension (constraints, interface, mutability, streaming, memory, distributed setting, etc.). For each challenge include:
   - Learning goal intent
   - What changed from the original problem
   - Why this change matters for design decisions

- Challenge: Print `foo`, `bar`, `baz` repeatedly (`n` cycles) in strict order with 3 threads.
  - Learning goal intent: Generalize two-party alternation to k-party token passing.
  - What changed from the original problem: Parties increased from 2 to 3.
  - Why this change matters for design decisions: Pairwise signaling becomes harder to reason about; you need a scalable state/turn index design.

- Challenge: Same `foo`/`bar` problem, but `printFoo` and `printBar` can block unpredictably and occasionally timeout.
  - Learning goal intent: Handle fault-aware synchronization.
  - What changed from the original problem: Side effects are no longer reliably fast/successful.
  - Why this change matters for design decisions: You must design timeout, cancellation, and invariant restoration paths.

- Challenge: Solve alternation across two processes (not threads) communicating via IPC queue.
  - Learning goal intent: Transfer local concurrency control to distributed boundary thinking.
  - What changed from the original problem: Shared memory primitives are unavailable.
  - Why this change matters for design decisions: Protocol correctness depends on message ordering, retries, and idempotency rather than in-process locks/events.



Yes. **`Lock.acquire()` is normally non-busy-waiting** in Python.

Example:

```python
import threading

lock = threading.Lock()

lock.acquire()
try:
    # critical section
    pass
finally:
    lock.release()
```

If another thread already holds the lock:

```python
lock.acquire()
```

will **block**. The thread is put to sleep by the runtime/OS synchronization primitive and wakes up when the lock becomes available.

It does **not** do:

```python
while lock_is_taken:
    check_again()
```

in user space.

---

## However: there is a caveat

Python exposes a non-blocking mode:

```python
if lock.acquire(blocking=False):
    try:
        work()
    finally:
        lock.release()
else:
    # failed immediately
    pass
```

This is polling if you wrap it:

```python
while not lock.acquire(blocking=False):
    pass
```

That is busy waiting.

---

## Timeline comparison

### Blocking lock

```
Thread A:
    acquire()
    |
    | holds lock
    |
    release()
              \
               wakes Thread B

Thread B:
    sleeps ----------------> runs
```

No CPU consumed while waiting.

---

### Spin lock (busy waiting)

```
Thread B:

while lock_taken:
    check
    check
    check
    check
```

CPU consumed while waiting.

---

## Relationship to `Condition`

A `Condition` is usually built on top of a lock:

```python
with condition:
    while not predicate():
        condition.wait()
```

The important part:

```python
condition.wait()
```

does **not** hold the lock while sleeping.

Conceptually:

$$
\text{Acquire Lock}
\rightarrow
\text{Check Predicate}
\rightarrow
\text{Release Lock + Sleep}
\rightarrow
\text{Wake}
\rightarrow
\text{Reacquire Lock}
\rightarrow
\text{Check Again}
$$

The "check again" loop exists because notifications can be spurious or the condition may have changed before reacquiring the lock.

---

## Rust analogy

Python:

```python
threading.Lock
```

is closest to:

```rust
std::sync::Mutex<T>
```

not:

```rust
std::sync::atomic::AtomicBool
```

A mutex blocks:

$$
Thread \rightarrow Sleeping
$$

An atomic operation does not block:

$$
Thread \rightarrow Continue \text{ after atomic instruction}
$$

A spin lock is the hybrid:

$$
Thread \rightarrow Busy\ Loop \rightarrow Retry
$$

So the hierarchy is:

$$
\text{Atomic} \neq \text{Lock} \neq \text{Condition}
$$

* **Atomic**: fastest state transition, no waiting mechanism.
* **Lock**: exclusive ownership, blocking when unavailable.
* **Condition**: blocking until a state predicate becomes true.
